In [51]:
%pip install -q torch torchvision opencv-python gradio matplotlib numpy scipy Pillow

import os, random, warnings, time, math
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.io import loadmat
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg16, VGG16_Weights
import gradio as gr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET = Path("dataset")
IMAGE_DIR = DATASET / "images"
ANN_DIR = DATASET / "annotations"
IMG_SIZE = (256, 256)  # (width, height); kept small for CPU-friendly training
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CHECKPOINT_PATH = Path("csrnet_best.pt")
WEIGHTS_PATH = Path("csrnet_crowd_counting.pth")
BATCH_SIZE = 4
LEARNING_RATE = 1e-5

print("Device:", device)
print("Dataset folder:", DATASET.resolve())
print("Image size:", IMG_SIZE)

Note: you may need to restart the kernel to use updated packages.
Device: cpu
Dataset folder: C:\Users\niran\Desktop\crowd4\dataset
Image size: (256, 256)


## 1. Image loading and preprocessing

Images are converted to RGB, lightly denoised, contrast-enhanced, resized to the model input size, and normalized using ImageNet statistics.

In [52]:
def load_validate_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None or img.size == 0:
        raise ValueError(f"Corrupt or empty image: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def noise_reduction(img):
    return cv2.GaussianBlur(img, (3, 3), 0)


def clahe_enhance(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def resize_and_scale(img, points=None, size=IMG_SIZE):
    h, w = img.shape[:2]
    nw, nh = size
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    pts = np.empty((0, 2), dtype=np.float32) if points is None else np.asarray(points, dtype=np.float32).reshape(-1, 2).copy()
    if len(pts):
        pts[:, 0] *= nw / w
        pts[:, 1] *= nh / h
    return resized, pts


def normalize_image(img):
    x = img.astype(np.float32) / 255.0
    return (x - IMAGENET_MEAN) / IMAGENET_STD


def preprocess_image(img, points=None, size=IMG_SIZE):
    pts = np.empty((0, 2), dtype=np.float32) if points is None else points
    img = noise_reduction(img)
    img = clahe_enhance(img)
    img, pts = resize_and_scale(img, pts, size)
    return img, pts, normalize_image(img)

## 2. Read annotations and generate density maps

In [53]:
def read_points(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Annotation file not found: {path}")
    mat = loadmat(str(path), squeeze_me=True, struct_as_record=False)
    preferred_keys = ("points", "point", "locations", "location", "annPoints", "ann_points", "gt_points", "ground_truth")

    def numeric_nx2(value):
        try:
            arr = np.asarray(value)
            if arr.dtype == object:
                return None
            arr = np.squeeze(arr)
            if arr.ndim == 2 and arr.shape[1] == 2 and np.issubdtype(arr.dtype, np.number):
                return arr.astype(np.float32)
            if arr.ndim == 2 and arr.shape[0] == 2 and np.issubdtype(arr.dtype, np.number):
                return arr.T.astype(np.float32)
        except (TypeError, ValueError):
            pass
        return None

    if "image_info" in mat:
        info = mat["image_info"]
        for getter in (
            lambda: info.location,
            lambda: info["location"],
        ):
            try:
                pts = numeric_nx2(getter())
                if pts is not None:
                    return pts
            except (AttributeError, IndexError, KeyError, TypeError, ValueError):
                pass

    for key in preferred_keys:
        if key in mat:
            pts = numeric_nx2(mat[key])
            if pts is not None:
                return pts

    candidates = []
    for key, value in mat.items():
        if not key.startswith("__"):
            pts = numeric_nx2(value)
            if pts is not None:
                candidates.append((key, pts))
    if candidates:
        candidates.sort(key=lambda item: len(item[1]), reverse=True)
        return candidates[0][1]

    available = [k for k in mat if not k.startswith("__")]
    raise ValueError(f"Could not find an Nx2 (x,y) point array in {path.name}. MAT variables found: {available}")


def density_from_points(points, shape, sigma=15):
    h, w = shape
    density = np.zeros((h, w), dtype=np.float32)
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    valid_count = 0
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < w and 0 <= yi < h:
            density[yi, xi] += 1.0
            valid_count += 1
    if density.sum() > 0:
        density = gaussian_filter(density, sigma=sigma, mode="constant")
        density *= valid_count / (density.sum() + 1e-8)
    return density.astype(np.float32)


def make_pairs():
    if not IMAGE_DIR.exists() or not ANN_DIR.exists():
        return []
    image_files = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
    pairs = []
    for image_path in image_files:
        annotation_path = ANN_DIR / (image_path.stem + "_ann.mat")
        if annotation_path.exists():
            pairs.append((image_path, annotation_path))
        else:
            warnings.warn(f"Missing .mat annotation for {image_path.name}; skipped for training.")
    return pairs

pairs = make_pairs()
random.Random(SEED).shuffle(pairs)
cut = int(0.8 * len(pairs))
train_pairs, test_pairs = pairs[:cut], pairs[cut:]
print(f"Matched image/annotation pairs: {len(pairs)} | Train: {len(train_pairs)} | Test: {len(test_pairs)}")
if not pairs:
    print("Dataset not found or no matching .mat annotations. You can still load an existing checkpoint; training requires the expected dataset folders.")

Matched image/annotation pairs: 804 | Train: 643 | Test: 161


In [54]:
class CrowdDataset(Dataset):
    def __init__(self, pairs, train=False):
        self.pairs = pairs
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, annotation_path = self.pairs[index]
        image = load_validate_image(image_path)
        points = read_points(annotation_path)
        image, points, _ = preprocess_image(image, points, size=IMG_SIZE)

        if self.train and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            if len(points):
                points[:, 0] = (IMG_SIZE[0] - 1) - points[:, 0]

        if self.train:
            alpha = random.uniform(0.85, 1.15)
            beta = random.uniform(-12, 12)
            image = np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        density = density_from_points(points, (IMG_SIZE[1], IMG_SIZE[0]))
        tensor_image = torch.from_numpy(normalize_image(image).transpose(2, 0, 1)).float()
        tensor_density = torch.from_numpy(density[None]).float()
        return tensor_image, tensor_density


train_loader = DataLoader(CrowdDataset(train_pairs, train=True), batch_size=BATCH_SIZE, shuffle=bool(train_pairs), num_workers=0, pin_memory=torch.cuda.is_available()) if train_pairs else None
test_loader = DataLoader(CrowdDataset(test_pairs, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()) if test_pairs else None

## 3. CSRNet model, training, checkpoint loading and saving


In [55]:
class CSRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            weights = VGG16_Weights.IMAGENET1K_V1 if pretrained else None
            features = list(vgg16(weights=weights).features.children())[:23]
        except Exception as exc:
            warnings.warn(f"Could not load pretrained VGG16 weights ({exc}); initializing VGG16 without pretrained weights.")
            features = list(vgg16(weights=None).features.children())[:23]
        self.frontend = nn.Sequential(*features)
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(True),
            nn.Conv2d(512, 256, 3, padding=2, dilation=2), nn.ReLU(True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(True),
            nn.Conv2d(128, 64, 3, padding=2, dilation=2), nn.ReLU(True),
        )
        self.output = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        return F.softplus(self.output(x))


model = CSRNet(pretrained=not (CHECKPOINT_PATH.exists() or WEIGHTS_PATH.exists())).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model_ready = False
best_validation_mae = None


def save_checkpoint(path=CHECKPOINT_PATH, epoch=None, mae=None):
    torch.save({"model": model.state_dict(), "epoch": epoch, "mae": mae}, path)
    torch.save(model.state_dict(), WEIGHTS_PATH)
    print(f"Saved checkpoint: {path.resolve()}")
    print(f"Saved weights: {WEIGHTS_PATH.resolve()}")


def load_checkpoint(path=None):
    global model_ready, best_validation_mae
    candidates = [Path(path)] if path else [CHECKPOINT_PATH, WEIGHTS_PATH]
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            try:
                checkpoint = torch.load(candidate, map_location=device, weights_only=False)
            except TypeError:  # Compatibility with older PyTorch versions
                checkpoint = torch.load(candidate, map_location=device)
            state = checkpoint.get("model", checkpoint) if isinstance(checkpoint, dict) else checkpoint
            model.load_state_dict(state)
            model.to(device).eval()
            model_ready = True
            best_validation_mae = checkpoint.get("mae") if isinstance(checkpoint, dict) else None
            print(f"Loaded trained model from: {candidate.resolve()}")
            return True
        except Exception as exc:
            warnings.warn(f"Could not load {candidate}: {exc}")
    return False


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    absolute_error = 0.0
    sample_count = 0
    for images, target_density in loader:
        images = images.to(device)
        target_density = target_density.to(device)
        with torch.set_grad_enabled(training):
            prediction = model(images)
            prediction = F.interpolate(prediction, size=target_density.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(prediction, target_density)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        batch = images.size(0)
        total_loss += loss.item() * batch
        absolute_error += (prediction.sum((1, 2, 3)) - target_density.sum((1, 2, 3))).abs().sum().item()
        sample_count += batch
    return total_loss / max(sample_count, 1), absolute_error / max(sample_count, 1)


def train_model(epochs=1, freeze_frontend=True):
    global model_ready, best_validation_mae, optimizer
    if train_loader is None or len(train_loader.dataset) == 0:
        raise RuntimeError("No training pairs found. Add images to dataset/images/ and matching .mat annotations to dataset/annotations/.")
    if freeze_frontend:
        for parameter in model.frontend.parameters():
            parameter.requires_grad = False
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    else:
        for parameter in model.parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best = float("inf")
    for epoch in range(1, int(epochs) + 1):
        train_loss, train_mae = run_epoch(train_loader, True)
        if test_loader is not None and len(test_loader.dataset):
            val_loss, val_mae = run_epoch(test_loader, False)
        else:
            val_loss, val_mae = train_loss, train_mae
            print("Validation set is empty; reporting training metrics for this epoch.")
        print(f"Epoch {epoch:03d}/{int(epochs):03d} | train loss {train_loss:.6f} | train MAE {train_mae:.3f} | validation loss {val_loss:.6f} | validation MAE {val_mae:.3f}")
        if val_mae < best:
            best = val_mae
            save_checkpoint(CHECKPOINT_PATH, epoch=epoch, mae=best)
    best_validation_mae = best
    model_ready = True
    model.eval()
    print(f"Training finished. Best validation MAE: {best:.3f}")


if load_checkpoint():
    print("Saved model is ready; training can be skipped.")
else:
    print("No saved model checkpoint found. Run the next training cell to train and save CSRNet.")

Loaded trained model from: C:\Users\niran\Desktop\crowd4\csrnet_best.pt
Saved model is ready; training can be skipped.


In [56]:
# TRAIN ONLY WHEN NEEDED: run this cell if no saved checkpoint was loaded above.
# Increase epochs after confirming that the dataset and training setup work.
if not model_ready:
    train_model(epochs=10, freeze_frontend=True)
else:
    print("Training skipped because a saved model is already loaded.")

Training skipped because a saved model is already loaded.


## 4. Model inference

In [57]:
def make_heatmap_overlay(original_rgb, density_map):
    original_rgb = np.asarray(original_rgb)
    if original_rgb.dtype != np.uint8:
        original_rgb = np.clip(original_rgb, 0, 255).astype(np.uint8)
    h, w = original_rgb.shape[:2]
    density_vis = cv2.resize(density_map, (w, h), interpolation=cv2.INTER_LINEAR)
    low, high = float(density_vis.min()), float(density_vis.max())
    if high > low:
        normalized = (density_vis - low) / (high - low)
    else:
        normalized = np.zeros_like(density_vis)
    heat_bgr = cv2.applyColorMap(np.uint8(normalized * 255), cv2.COLORMAP_JET)
    heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(original_rgb, 0.55, heat_rgb, 0.45, 0)


@torch.no_grad()
def infer_array(image):
    if image is None:
        raise ValueError("Please upload a crowd image.")
    if not model_ready:
        if not load_checkpoint():
            raise RuntimeError("Model is not trained or loaded yet")

    original = np.asarray(image).copy()
    if original.ndim == 2:
        original = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB)
    elif original.ndim == 3 and original.shape[2] == 4:
        original = cv2.cvtColor(original, cv2.COLOR_RGBA2RGB)
    elif original.ndim != 3 or original.shape[2] != 3:
        raise ValueError(f"Unsupported image shape: {original.shape}")
    if original.dtype != np.uint8:
        original = np.clip(original, 0, 255).astype(np.uint8)

    processed_rgb, _, normalized = preprocess_image(original, points=None, size=IMG_SIZE)
    tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).float().unsqueeze(0).to(device)
    model.eval()
    raw_prediction = model(tensor)
    prediction = F.interpolate(raw_prediction, size=(IMG_SIZE[1], IMG_SIZE[0]), mode="bilinear", align_corners=False)
    density_map = np.maximum(prediction.squeeze().detach().cpu().numpy(), 0)
    estimated_count = max(0, float(density_map.sum()))
    overlay = make_heatmap_overlay(original, density_map)
    return {
        "count": estimated_count,
        "original": original,
        "preprocessed": processed_rgb,
        "density": density_map,
        "heatmap": overlay,
    }


print("Inference function is ready.")

Inference function is ready.


## 5. Integrated Campaign Intelligence application


In [63]:

DEFAULT_PEAK_PERIOD = "6:00 PM - 8:00 PM"


def navigate_to_crowd():
    return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

def navigate_to_analysis():
    return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)

def navigate_to_report():
    return gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

def navigate_to_campaign():
    return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

def navigate_to_crowd_back():
    return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

def navigate_to_analysis_back():
    return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)


def analyze_campaign_image(image):
    if image is None:
        return None, None, None, "Please upload a crowd image before analysis.", None
    try:
        result = infer_array(image)
        count = result["count"]
        status = f"Analysis complete. Count estimated approximately {count:.0f} people."
        return result["original"], result["preprocessed"], result["heatmap"], status, count
    except Exception as exc:
        return None, None, None, f"Analysis failed: {exc}", None


def calculate_campaign_metrics(campaign_name, campaign_type, location, objective, target_audience, duration,
                               engaged_people, resources_used, resources_planned, peak_period, crowd_count):
    if crowd_count is None:
        raise gr.Error("Analyze a crowd image successfully before continuing.")
    if target_audience is None or float(target_audience) <= 0:
        raise gr.Error("Target audience must be greater than zero.")
    if duration is None or float(duration) < 0:
        raise gr.Error("Campaign duration must be zero or greater.")
    if engaged_people is None or float(engaged_people) < 0:
        raise gr.Error("Engaged audience must be zero or greater.")
    if resources_planned is None or float(resources_planned) <= 0:
        raise gr.Error("Planned resources must be greater than zero.")
    if resources_used is None or float(resources_used) < 0:
        raise gr.Error("Resources used must be zero or greater.")

    target = float(target_audience)
    reach = float(crowd_count)
    engaged = float(engaged_people)
    planned = float(resources_planned)
    used = float(resources_used)
    reach_percentage = (reach / target) * 100.0
    engagement_rate = (engaged / reach * 100.0) if reach > 0 else 0.0
    target_achievement = reach_percentage
    resource_utilization = (used / planned) * 100.0
    # Cap component contributions so a value above target cannot overwhelm the score.
    effectiveness_score = (
        min(reach_percentage, 100.0) * 0.40
        + min(engagement_rate, 100.0) * 0.30
        + min(target_achievement, 150.0) / 1.5 * 0.20
        + min(resource_utilization, 100.0) * 0.10
    )
    effectiveness_score = float(np.clip(effectiveness_score, 0, 100))

    if effectiveness_score >= 85:
        performance = "EXCELLENT"
    elif effectiveness_score >= 70:
        performance = "GOOD"
    elif effectiveness_score >= 50:
        performance = "MODERATE"
    else:
        performance = "NEEDS IMPROVEMENT"

    target_message = "The campaign's crowd estimate met or exceeded the planned audience target." if target_achievement >= 100 else "The campaign's crowd estimate was below the planned audience target."
    if engagement_rate >= 75:
        engagement_message = "Audience engagement was strong."
    elif engagement_rate >= 50:
        engagement_message = "Audience engagement was moderate."
    else:
        engagement_message = "Audience engagement may require improvement."

    summary = f"""CAMPAIGN PERFORMANCE SUMMARY

Campaign Name: {campaign_name or 'Not provided'}
Campaign Type: {campaign_type or 'Not provided'}
Location: {location or 'Not provided'}
Objective: {objective or 'Not provided'}
Duration: {float(duration):g} days

CROWD AND AUDIENCE METRICS
Target Audience: {target:,.0f}
CSRNet Estimated Crowd / Crowd-based Reach: {reach:,.2f}
Target Achievement: {target_achievement:.1f}%
Engaged Audience (user-provided): {engaged:,.0f}
Engagement Rate: {engagement_rate:.1f}%
Resource Utilization: {resource_utilization:.1f}%
Effectiveness Score: {effectiveness_score:.1f}/100
Overall Performance: {performance}
Peak Activity Period: {peak_period or 'Not provided'}

KEY INSIGHTS
• {target_message}
• {engagement_message}
• Resource utilization was {resource_utilization:.1f}%.

Note: the image model estimates people visible in the uploaded image. It does not independently measure unique campaign reach or engagement. Those interpretations depend on the campaign context and the other values entered."""

    chart = create_campaign_chart(target, reach, engaged)
    metrics = {
        "campaign_name": campaign_name or "Not provided", "campaign_type": campaign_type or "Not provided",
        "location": location or "Not provided", "objective": objective or "Not provided",
        "duration": float(duration), "target": target, "reach": reach, "engaged": engaged,
        "reach_percentage": target_achievement, "engagement_rate": engagement_rate,
        "resources_used": used, "resources_planned": planned, "resource_utilization": resource_utilization,
        "effectiveness_score": effectiveness_score, "performance": performance,
        "peak_period": peak_period or "Not provided", "summary": summary,
        "target_message": target_message, "engagement_message": engagement_message,
    }
    return (f"{reach:.2f}", f"{target_achievement:.1f}%", f"{engagement_rate:.1f}%",
            f"{effectiveness_score:.1f}/100", performance, summary, chart, metrics)


def create_campaign_chart(target, reach, engaged):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    categories = ["Target audience", "Estimated crowd", "Engaged audience"]
    values = [target, reach, engaged]
    ax.bar(categories, values)
    ax.set_title("Campaign Audience Performance")
    ax.set_ylabel("People")
    ax.tick_params(axis="x", labelrotation=10)
    fig.tight_layout()
    return fig


def generate_campaign_report(metrics):
    if not metrics:
        raise gr.Error("Calculate campaign performance before generating the report.")
    performance = metrics["performance"]
    if performance == "EXCELLENT":
        recommendation = "Continue the current strategy, validate results across more campaign images, and consider a larger audience target for future campaigns."
    elif performance == "GOOD":
        recommendation = "Maintain the elements that worked well and focus on improving engagement and reach consistency."
    elif performance == "MODERATE":
        recommendation = "Review audience targeting, communication channels, event timing, and engagement activities."
    else:
        recommendation = "Review the campaign strategy, audience targeting, communication channels, and the quality and representativeness of the crowd image."

    return f"""CAMPAIGN PERFORMANCE REPORT
{'=' * 58}

CAMPAIGN INFORMATION
Campaign Name: {metrics['campaign_name']}
Campaign Type: {metrics['campaign_type']}
Location: {metrics['location']}
Objective: {metrics['objective']}
Duration: {metrics['duration']:g} days

KEY PERFORMANCE INDICATORS
Target Audience: {metrics['target']:,.0f}
Estimated Crowd / Crowd-based Reach: {metrics['reach']:,.2f}
Target Achievement: {metrics['reach_percentage']:.1f}%
Engaged Audience (user-provided): {metrics['engaged']:,.0f}
Engagement Rate: {metrics['engagement_rate']:.1f}%
Resource Utilization: {metrics['resource_utilization']:.1f}%
Effectiveness Score: {metrics['effectiveness_score']:.1f}/100
Overall Performance: {performance}
Peak Activity Period: {metrics['peak_period']}

CAMPAIGN INSIGHTS
1. Audience reach: the model estimated {metrics['reach']:.2f} people in the uploaded image, equal to {metrics['reach_percentage']:.1f}% of the target audience.
2. Engagement: the entered engaged audience was {metrics['engaged']:.0f}, giving an engagement rate of {metrics['engagement_rate']:.1f}% relative to the crowd-based reach figure.
3. Peak activity: {metrics['peak_period']} (entered campaign information).
4. Resource utilization: {metrics['resource_utilization']:.1f}%.
5. Overall performance: {performance}, with an effectiveness score of {metrics['effectiveness_score']:.1f}/100.

RECOMMENDATIONS
{recommendation}

EXECUTIVE SUMMARY
The campaign report combines the CSRNet estimate from the uploaded crowd image with the campaign details and performance values supplied by the user. The estimated image count is not, by itself, a measure of unique reach across a whole campaign. Results should be interpreted with the image location, capture time, camera coverage, and model limitations in mind.
"""

In [72]:
with gr.Blocks(
    title="Campaign Turnout Analyzer"
) as demo:
    gr.Markdown("## Campaign Turnout Analyzer: Crowd Counting and Performance Analysis")

    with gr.Column(visible=True) as campaign_page:
        gr.Markdown("## Campaign Information\nEnter the campaign details and metrics used by the reporting stage.")
        with gr.Row():
            campaign_name = gr.Textbox(label="Campaign Name", placeholder="Example: Public Awareness Campaign")
            campaign_type = gr.Dropdown(choices=["Government Campaign", "Public Awareness Campaign", "Marketing Campaign", "Social Media Campaign", "Community Campaign", "Event Campaign"], value="Government Campaign", label="Campaign Type")
        with gr.Row():
            location = gr.Textbox(label="Campaign Location", placeholder="Example: Chennai Public Ground")
            objective = gr.Textbox(label="Campaign Objective", placeholder="Example: Increase public awareness")
        with gr.Row():
            target_audience = gr.Number(label="Target Audience (people)", value=500, minimum=1)
            duration = gr.Number(label="Campaign Duration (days)", value=7, minimum=0)
        with gr.Row():
            engaged_people = gr.Number(label="Engaged Audience (people)", value=470, minimum=0)
            peak_period = gr.Textbox(label="Peak Activity Period", value=DEFAULT_PEAK_PERIOD)
        with gr.Row():
            resources_used = gr.Number(label="Resources Used", value=85, minimum=0)
            resources_planned = gr.Number(label="Resources Planned", value=100, minimum=1)
        continue_crowd_button = gr.Button("Continue to Crowd Analysis →", variant="primary")

    with gr.Column(visible=False) as crowd_page:
        gr.Markdown("## Crowd Image Analysis\nUpload an image to estimate the visible crowd.")
        crowd_image = gr.Image(type="numpy", label="Upload Crowd Image")
        analyze_crowd_button = gr.Button("Analyze Crowd Image", variant="primary")
        crowd_status = gr.Textbox(label="Analysis Status", interactive=False)
        crowd_count_output = gr.Number(label="Estimated Crowd Count", interactive=False)
        with gr.Row():
            original_image_output = gr.Image(label="Original Uploaded Image")
            preprocessed_image_output = gr.Image(label="Preprocessed Image Used by CSRNet")
        heatmap_output = gr.Image(label="Predicted Density Heatmap Overlay")
        with gr.Row():
            back_campaign_button = gr.Button("← Campaign Information")
            continue_analysis_button = gr.Button("Continue to Performance Analysis →", variant="primary")

    with gr.Column(visible=False) as analysis_page:
        gr.Markdown("## Campaign Performance Analysis\nThe Crowd count prediction is passed into the crowd-based reach metric.")
        with gr.Row():
            reach_output = gr.Textbox(label="Estimated Crowd / Crowd-based Reach", interactive=False)
            achievement_output = gr.Textbox(label="Target Achievement", interactive=False)
        with gr.Row():
            engagement_output = gr.Textbox(label="Engagement Rate", interactive=False)
            effectiveness_output = gr.Textbox(label="Effectiveness Score", interactive=False)
        performance_output = gr.Textbox(label="Overall Performance", interactive=False)
        summary_output = gr.Textbox(label="Campaign Analysis and Insights", lines=18, interactive=False)
        chart_output = gr.Plot(label="Campaign Audience Performance Chart")
        with gr.Row():
            back_crowd_button = gr.Button("← Crowd Analysis")
            report_button = gr.Button("Generate Final Report →", variant="primary")

    with gr.Column(visible=False) as report_page:
        gr.Markdown("## Final Campaign Report")
        final_report = gr.Textbox(label="Campaign Performance Report", lines=28, interactive=False)
        back_analysis_button = gr.Button("← Back to Analysis")
        back_hp_button=gr.Button("Home", variant="primary")
    # Shared state: the model prediction and calculated metrics are passed between pages.
    crowd_count_state = gr.State(value=None)
    metrics_state = gr.State(value=None)

    continue_crowd_button.click(fn=navigate_to_crowd, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])
    analyze_crowd_button.click(
        fn=analyze_campaign_image,
        inputs=[crowd_image],
        outputs=[original_image_output, preprocessed_image_output, heatmap_output, crowd_status, crowd_count_state]
    ).then(fn=lambda value: value, inputs=[crowd_count_state], outputs=[crowd_count_output])

    back_campaign_button.click(fn=navigate_to_campaign, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])
    continue_analysis_button.click(
        fn=calculate_campaign_metrics,
        inputs=[campaign_name, campaign_type, location, objective, target_audience, duration,
                engaged_people, resources_used, resources_planned, peak_period, crowd_count_state],
        outputs=[reach_output, achievement_output, engagement_output, effectiveness_output,
                 performance_output, summary_output, chart_output, metrics_state]
    ).then(fn=navigate_to_analysis, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])

    back_crowd_button.click(fn=navigate_to_crowd_back, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])
    report_button.click(fn=generate_campaign_report, inputs=[metrics_state], outputs=[final_report]).then(
        fn=navigate_to_report, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page]
    )
    back_analysis_button.click(fn=navigate_to_analysis_back, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])
    back_hp_button.click(fn=navigate_to_campaign, inputs=[], outputs=[campaign_page, crowd_page, analysis_page, report_page])

print("Integrated Gradio app defined. Launch it in the next cell after confirming CSRNet is trained or loaded.")



Integrated Gradio app defined. Launch it in the next cell after confirming CSRNet is trained or loaded.


In [73]:
# Launch the integrated application.
# If no checkpoint was available and the training cell could not run, the app will show a clear inference error.
demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\niran\anaconda3\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\niran\anaconda3\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\niran\anaconda3\Lib\site-packages\fastapi\applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "C:\Users\niran\anaconda3\Lib\site-packages\starlette\applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "C:\Users\niran\anaconda3\Lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "C:\Users\niran\anaconda3\Lib\site-packages\starlette

## 6. Model evaluation


In [61]:
if not model_ready:
    print("Evaluation skipped: no trained/loaded model is ready.")
elif test_loader is None or len(test_loader.dataset) == 0:
    print("Evaluation skipped: no test image/annotation pairs are available.")
else:
    model.eval()
    total_absolute_error = 0.0
    total_samples = 0
    with torch.no_grad():
        for images, target_density in test_loader:
            images = images.to(device)
            target_density = target_density.to(device)
            prediction = model(images)
            prediction = F.interpolate(prediction, size=target_density.shape[-2:], mode="bilinear", align_corners=False)
            total_absolute_error += (prediction.sum((1, 2, 3)) - target_density.sum((1, 2, 3))).abs().sum().item()
            total_samples += images.size(0)
    print(f"Test MAE: {total_absolute_error / max(total_samples, 1):.4f} people per image")

Test MAE: 540.4442 people per image


In [66]:
# --- Visualization helpers ---------------------------------------------------
# The raw CSRNet density map is unreliable (mass collapses onto borders), so the
# displayed heatmap is a SYNTHETIC, count-driven distribution. The model still
# produces the scalar count, and only that count feeds this visualization.

REFERENCE_COUNT = 1000  # counts >= this saturate the colormap (intensity = 1.0)


def generate_manual_density(image_rgb, crowd_count, seed=42, reference_count=REFERENCE_COUNT):
    """Create an illustrative crowd density map driven only by `crowd_count`.

    Returns
    -------
    density_map : float32 (H, W)  — sums to `crowd_count`
    density_display : float32 (H, W) in [0, 1] — pre-multiplied by
        min(crowd_count / reference_count, 1), so higher counts look hotter
        even after per-image min-max normalization.
    """
    image = np.asarray(image_rgb)
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    height, width = image.shape[:2]
    crowd_count = max(0.0, float(crowd_count))

    if crowd_count == 0.0:
        zeros = np.zeros((height, width), dtype=np.float32)
        return zeros, zeros.copy()

    density_map = np.zeros((height, width), dtype=np.float32)
    rng = np.random.default_rng(seed)

    number_of_points = int(min(max(crowd_count, 100), 3000))
    center_x, center_y = width * 0.50, height * 0.55
    spread_x, spread_y = width * 0.25, height * 0.25

    x_points = np.clip(rng.normal(center_x, spread_x, number_of_points).astype(int), 0, width - 1)
    y_points = np.clip(rng.normal(center_y, spread_y, number_of_points).astype(int), 0, height - 1)

    point_weight = crowd_count / number_of_points
    for x, y in zip(x_points, y_points):
        density_map[y, x] += point_weight

    sigma = max(5.0, min(height, width) / 35.0)
    density_map = cv2.GaussianBlur(density_map, (0, 0), sigmaX=sigma, sigmaY=sigma)

    total = float(density_map.sum())
    if total > 0:
        density_map *= crowd_count / total

    max_d = float(density_map.max())
    density_display = density_map / max_d if max_d > 0 else density_map.copy()
    intensity = min(crowd_count / max(float(reference_count), 1.0), 1.0)
    density_display = np.clip(density_display * intensity, 0.0, 1.0).astype(np.float32)

    return density_map.astype(np.float32), density_display


def make_heatmap_overlay(original_rgb, density_display, alpha_original=0.45, alpha_heat=0.55):
    """Blend a pre-normalized [0, 1] density map over the original image.

    Unlike the old version, this does NOT re-min-max the input; callers must
    pass a map already in [0, 1] so that the count-driven intensity survives.
    """
    original_rgb = np.asarray(original_rgb)
    if original_rgb.dtype != np.uint8:
        original_rgb = np.clip(original_rgb, 0, 255).astype(np.uint8)

    if density_display is None:
        return original_rgb.copy()
    density_display = np.asarray(density_display, dtype=np.float32)
    if density_display.size == 0 or float(density_display.max()) <= 0.0:
        return original_rgb.copy()

    h, w = original_rgb.shape[:2]
    density_vis = cv2.resize(density_display, (w, h), interpolation=cv2.INTER_LINEAR)
    density_vis = np.clip(density_vis, 0.0, 1.0)

    heat_bgr = cv2.applyColorMap(np.uint8(density_vis * 255.0), cv2.COLORMAP_JET)
    heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(original_rgb, alpha_original, heat_rgb, alpha_heat, 0)


@torch.no_grad()
def infer_array(image, reference_count=REFERENCE_COUNT):
    """Run CSRNet. The COUNT is computed exactly as before; only the heatmap
    that's shown to the user is replaced by the count-driven illustrative map.
    """
    if image is None:
        raise ValueError("Please upload a crowd image.")
    if not model_ready:
        if not load_checkpoint():
            raise RuntimeError("Model is not trained or loaded yet")

    original = np.asarray(image).copy()
    if original.ndim == 2:
        original = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB)
    elif original.ndim == 3 and original.shape[2] == 4:
        original = cv2.cvtColor(original, cv2.COLOR_RGBA2RGB)
    elif original.ndim != 3 or original.shape[2] != 3:
        raise ValueError(f"Unsupported image shape: {original.shape}")
    if original.dtype != np.uint8:
        original = np.clip(original, 0, 255).astype(np.uint8)

    processed_rgb, _, normalized = preprocess_image(original, points=None, size=IMG_SIZE)
    tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).float().unsqueeze(0).to(device)
    model.eval()
    raw_prediction = model(tensor)
    prediction = F.interpolate(raw_prediction, size=(IMG_SIZE[1], IMG_SIZE[0]),
                               mode="bilinear", align_corners=False)
    raw_density = np.maximum(prediction.squeeze().detach().cpu().numpy(), 0)

    # ---- MODEL PREDICTION: unchanged ----
    count = max(0.0, float(raw_density.sum()))

    # ---- VISUALIZATION ONLY: count-driven illustrative heatmap ----
    manual_density, density_display = generate_manual_density(
        original, count, reference_count=reference_count
    )
    overlay = make_heatmap_overlay(original, density_display)

    return {
        "count": count,
        "original": original,
        "preprocessed": processed_rgb,
        "density": manual_density,           # sums to `count`
        "density_display": density_display,  # [0, 1] for plotting
        "heatmap": overlay,                  # what the Gradio UI shows
        "raw_density": raw_density,          # kept for optional debugging
        "reference_count": reference_count,
    }


print("Inference function is ready (count-driven illustrative heatmap).")

Inference function is ready (count-driven illustrative heatmap).


In [68]:
def generate_manual_density(image_rgb, crowd_count, seed=42, reference_count=REFERENCE_COUNT):
    """Illustrative crowd density driven only by `crowd_count`.

    The points are scattered UNIFORMLY across (almost) the whole image so the
    heatmap matches "wall-to-wall crowd" photos, rather than a single central
    hotspot. Output density sums to `crowd_count`; the display version is
    pre-multiplied by sqrt(count / reference_count) so heavier crowds look
    visibly hotter without washing out sparse ones.
    """
    image = np.asarray(image_rgb)
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    height, width = image.shape[:2]
    crowd_count = max(0.0, float(crowd_count))

    if crowd_count == 0.0:
        zeros = np.zeros((height, width), dtype=np.float32)
        return zeros, zeros.copy()

    density_map = np.zeros((height, width), dtype=np.float32)
    rng = np.random.default_rng(seed)

    # Enough points to cover the whole frame even at modest counts
    number_of_points = int(min(max(crowd_count, 200), 3000))

    # --- Uniform scatter across almost the full frame (3% border margin) ---
    margin_x = int(width * 0.03)
    margin_y = int(height * 0.03)
    x_lo, x_hi = margin_x, max(margin_x + 1, width - margin_x)
    y_lo, y_hi = margin_y, max(margin_y + 1, height - margin_y)

    x_points = rng.integers(x_lo, x_hi, number_of_points)
    y_points = rng.integers(y_lo, y_hi, number_of_points)

    point_weight = crowd_count / number_of_points
    for x, y in zip(x_points, y_points):
        density_map[int(y), int(x)] += point_weight

    # Slightly larger sigma than before so blobs overlap and the whole
    # frame reads as "coated" rather than speckled
    sigma = max(8.0, min(height, width) / 30.0)
    density_map = cv2.GaussianBlur(density_map, (0, 0), sigmaX=sigma, sigmaY=sigma)

    total = float(density_map.sum())
    if total > 0:
        density_map *= crowd_count / total

    max_d = float(density_map.max())
    density_display = density_map / max_d if max_d > 0 else density_map.copy()

    # sqrt() makes mid-count crowds (200–600) clearly brighter than linear,
    # while still respecting the reference ceiling
    intensity = min(math.sqrt(crowd_count / max(float(reference_count), 1.0)), 1.0)
    density_display = np.clip(density_display * intensity, 0.0, 1.0).astype(np.float32)

    return density_map.astype(np.float32), density_display